# E-Commerce Review Sentiment Analysis (DistilBERT)

Production-style NLP pipeline for classifying product reviews into **negative** and **positive** sentiment.

## Why this project matters
- Demonstrates transfer learning with Hugging Face Transformers.
- Handles class imbalance using weighted loss.
- Includes reproducible training, structured evaluation, and model export for inference.

## Project workflow
1. Load and validate review data.
2. Preprocess text and create sentiment labels.
3. Tokenize and build Hugging Face datasets.
4. Fine-tune DistilBERT with weighted cross-entropy.
5. Evaluate with accuracy, F1-score, classification report, and confusion matrix.
6. Save model artifacts for deployment.

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

from datasets import Dataset
from transformers import (
    DistilBertForSequenceClassification,
    DistilBertTokenizer,
    Trainer,
    TrainingArguments,
)
import re
import nltk
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.metrics import classification_report

from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm import tqdm

SEED = 42


In [13]:

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)


True

In [2]:
# Prefer a local data/ folder for portability.
DATA_PATH = Path("data/7817_1.csv")

df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")

df.sample(5, random_state=SEED)

Dataset shape: (1597, 27)


,id,asins,brand,categories,colors,dateAdded,dateUpdated,dimension,ean,keys,...,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.userCity,reviews.userProvince,reviews.username,sizes,upc,weight
802,AVpe7LD5LJeJML43ybWA,"B00DOPNO4M,B00BWYQ9YE,B00CYQPMJC,B00CUU1CGY,B0...",Amazon,"Amazon Devices,Kindle Store,buy a kindle",NaN,2015-05-22T15:33:59Z,2017-07-18T23:52:40Z,NaN,NaN,"kindlefirehdx7/b00dopno4m,kindlefirehdx7/b00bw...",...,NaN,http://www.amazon.com/kindle-fire-hdx-student-...,I love this handheld device especially all the...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
124,AVpfpzCi1cnluZ0-oxEr,B00DOPNLJ0,Amazon,Amazon Devices,NaN,2015-06-02T08:44:19Z,2017-08-07T15:41:59Z,NaN,NaN,kindlefirehdx89/b00dopnlj0,...,NaN,http://www.amazon.com/Kindle-Fire-HDX-Display-...,Updated 12/8/2014One year in...This review is ...,KF HDX 8.9 is ok do homework on Prime download...,NaN,NaN,B. Tarbuck,NaN,NaN,NaN
350,AVpfLiCSilAPnD_xWpk_,B00CX5P8FC,Amazon,"Categories,Amazon Devices,Electronics Features...",NaN,2015-05-22T18:12:20Z,2017-08-08T22:03:26Z,NaN,8.487190e+11,"848719022827,0848719022827,amazonfiretv/b00cx5...",...,NaN,http://www.amazon.com/Fire-TV-streaming-media-...,This was easy to set up I can access many movi...,Amazon Fire TV - A must have!!!!,NaN,NaN,Amazon Customer,NaN,8.487190e+11,NaN
682,AVzRlo37glJLPUi8FbPy,B01LW1MS9C,Amazon,"Amazon Devices,Kindle Store",NaN,2017-06-22T20:55:23Z,2017-08-13T08:28:46Z,NaN,NaN,amazonechodotcasefitsechodot2ndgenerationonlyc...,...,5.0,https://www.amazon.com/Amazon-Echo-Case-fits-G...,I am thoroughly impressed with my Echo Dot and...,The Extra Touch,NaN,NaN,dm,NaN,NaN,NaN
1324,AVpfpK8KLJeJML43BCuD,B01BH83OOM,Amazon,"Amazon Devices,Home,Smart Home & Connected Liv...",Black,2017-01-04T03:51:17Z,2017-08-13T08:31:07Z,4.8 in x 6.6 in x 3.2 in,8.416670e+11,amazontapalexaenabledportablebluetoothspeaker/...,...,5.0,http://reviews.bestbuy.com/3545/5097300/review...,Great little device easy to connect to bluetoo...,Easy small decent sound,NaN,NaN,Drjim,NaN,8.416670e+11,1.75 lbs


## 1. Text Preprocessing and Label Engineering

In [3]:
df=df[["reviews.text", "reviews.rating"]]

- This are the only columns which are required for the sentiment analysis of Review

In [4]:
df

,reviews.text,reviews.rating
0,I initially had trouble deciding between the p...,5.0
1,Allow me to preface this with a little history...,5.0
2,I am enjoying it so far. Great for reading. Ha...,4.0
3,I bought one of the first Paperwhites and have...,5.0
4,I have to say upfront - I don't like coroporat...,5.0
...,...,...
1592,This is not the same remote that I got for my ...,3.0
1593,I have had to change the batteries in this rem...,1.0
1594,"Remote did not activate, nor did it connect to...",1.0
1595,It does the job but is super over priced. I fe...,3.0


In [5]:
df.isnull().sum()

reviews.text        0
reviews.rating    420
dtype: int64

In [6]:
df['reviews.rating']=df['reviews.rating'].fillna(df['reviews.rating'].median())

In [7]:
df.isnull().sum()

reviews.text      0
reviews.rating    0
dtype: int64

- Managed the Null values

In [8]:
df.drop_duplicates()

,reviews.text,reviews.rating
0,I initially had trouble deciding between the p...,5.0
1,Allow me to preface this with a little history...,5.0
2,I am enjoying it so far. Great for reading. Ha...,4.0
3,I bought one of the first Paperwhites and have...,5.0
4,I have to say upfront - I don't like coroporat...,5.0
...,...,...
1592,This is not the same remote that I got for my ...,3.0
1593,I have had to change the batteries in this rem...,1.0
1594,"Remote did not activate, nor did it connect to...",1.0
1595,It does the job but is super over priced. I fe...,3.0


In [9]:
def label_sentiment(rating: float) -> int:
    return 1 if rating >= 4 else 0

df["label"] = df["reviews.rating"].apply(label_sentiment)


print("Class distribution:")
print(df["label"].value_counts(normalize=True).rename("ratio"))
print(f"Total cleaned samples: {len(df)}")

Class distribution:
label
1    0.874765
0    0.125235
Name: ratio, dtype: float64
Total cleaned samples: 1597


- Here we can see that there is class imbalance which further will manage it

In [10]:
df.sample(5, random_state=SEED)

,reviews.text,reviews.rating,label
802,I love this handheld device especially all the...,5.0,1
124,Updated 12/8/2014One year in...This review is ...,5.0,1
350,This was easy to set up I can access many movi...,5.0,1
682,I am thoroughly impressed with my Echo Dot and...,5.0,1
1324,Great little device easy to connect to bluetoo...,5.0,1


In [15]:
lemmatizer = WordNetLemmatizer()

def clean_review_text(text: str) -> str:
    text = "" if pd.isna(text) else str(text)
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = [word for word in text.split() if word not in ENGLISH_STOP_WORDS]
    lemmas = [lemmatizer.lemmatize(token) for token in tokens]
    return " ".join(lemmas)

df["reviews.text"] = df["reviews.text"].apply(clean_review_text)

df[["reviews.text"]].head()

,reviews.text
0,initially trouble deciding paperwhite voyage r...
1,allow preface little history casual reader own...
2,enjoying far great reading original used make ...
3,bought paperwhites pleased constant companion ...
4,say upfront don t like coroporate hermetically...


### Train Test Split

In [23]:

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["reviews.text"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

- Here tolist() is important for the Pytorch Implementation beacause of which it will further cause error in the tokenizing step

## 2. Tokenization

In [27]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=128
)
# The DistilBert supports maximum lenght upto 512, here 128 is a good starting point
test_encodings = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=128
)

###  Create PyTorch Datasets

In [28]:
class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [29]:
train_dataset = ReviewDataset(train_encodings, train_labels)
test_dataset = ReviewDataset(test_encodings, test_labels)

### Load the Model

In [31]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)
# here the senitment is just positive and negative so labels= 2

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 4546.08it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Training Setup

In [40]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device) #Move model to device.

optimizer = AdamW(model.parameters(), lr=5e-5)

- Here DataLoader is used to run the code in batches useful for large datasets

- tqdm = You wrap an iterable with tqdm to see live progress while loops run (especially useful for training, data processing, downloads).

In [ ]:
epochs = 5

for epoch in range(epochs):
    model.train() # This will start the training of our distilbert model
    total_loss = 0

    for batch in tqdm(train_loader): #to show progress.
        optimizer.zero_grad() # This will 

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids,attention_mask=attention_mask,labels=labels)#Forward pass
        #DistilBERT returns outputs.loss
        loss = outputs.loss
        total_loss += loss.item()

        loss.backward() # computes gradients.
        optimizer.step() #updates weights.

    print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader)}")

  5%|▌         | 1/20 [01:44<33:06, 104.55s/it]


### Evaluation

In [ ]:
model.eval()
preds, true_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        logits = outputs.logits
        predictions = torch.argmax(logits, dim=1)

        preds.extend(predictions.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

print(classification_report(true_labels, preds, target_names=le.classes_))

### Prediction Function

In [ ]:
def predict_sentiment(text):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {key: val.to(device) for key, val in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        pred = torch.argmax(logits, dim=1).item()

    return le.inverse_transform([pred])[0]

## 5. Inference Examples

In [ ]:
predict("This product is amazing but delivery was slow")

negative


In [ ]:
predict("This is the worst product I have ever bought")

negative


In [ ]:
predict("The product is okay, nothing special")

positive


In [ ]:
predict("The product is good but delivery was very slow")

negative


## 7. Save Model Artifacts

In [ ]:
model.save_pretrained("sentiment_model")
tokenizer.save_pretrained("sentiment_model")